In [41]:
! pip install hijri-converter

## Step 1:Import libraries


In [42]:
import os 
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
os.makedirs("results/eda", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

## Step 2: Data Exploration & Cleaning

In [43]:
train=pd.read_csv(r'data\train.csv',parse_dates=['Date'])
features = pd.read_csv(r"data/features.csv", parse_dates=["Date"])
stores = pd.read_csv(r"data/stores.csv")

In [44]:
print('Train:', train.shape)
print('Features:', features.shape)
print('Stores:', stores.shape)

Train: (421570, 5)
Features: (8190, 12)
Stores: (45, 3)


In [45]:
print(train.columns)
print(features.columns)
print(stores.columns)

Index(['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday'], dtype='object')
Index(['Store', 'Date', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2',
       'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment',
       'IsHoliday'],
      dtype='object')
Index(['Store', 'Type', 'Size'], dtype='object')


In [46]:
# features has IsHoliday too — drop the duplicate before merging to avoid a suffix clash
df = train.merge(features.drop(columns=["IsHoliday"]), on=["Store", "Date"], how="left")
df = df.merge(stores, on="Store", how="left")

In [47]:
print("\nmerged shape:", df.shape)


merged shape: (421570, 16)


In [48]:
df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


In [49]:
numerical_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "bool"]).columns.tolist()
print(f"\nNumerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")


Numerical columns (13): ['Store', 'Dept', 'Weekly_Sales', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Size']
Categorical columns (2): ['IsHoliday', 'Type']


In [50]:
n_rows, n_cols = df.shape
print(f"\nRows: {n_rows}  (need >= 10,000: {n_rows >= 10_000})")
print(f"Columns: {n_cols}  (need >= 10: {n_cols >= 10})")

print("\nDtypes:\n", df.dtypes)


Rows: 421570  (need >= 10,000: True)
Columns: 16  (need >= 10: True)

Dtypes:
 Store                    int64
Dept                     int64
Date            datetime64[ns]
Weekly_Sales           float64
IsHoliday                 bool
Temperature            float64
Fuel_Price             float64
MarkDown1              float64
MarkDown2              float64
MarkDown3              float64
MarkDown4              float64
MarkDown5              float64
CPI                    float64
Unemployment           float64
Type                    object
Size                     int64
dtype: object


In [79]:
for col in categorical_cols:
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-'*50)

IsHoliday
2
[0 1]
--------------------------------------------------
Type
3
['A' 'B' 'C']
--------------------------------------------------


In [80]:
for col in numerical_cols:
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-'*50)

Store
45
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45]
--------------------------------------------------
Dept
81
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 16 17 18 19 20 21 22 23 24 25
 26 27 28 29 30 31 32 33 34 35 36 37 38 40 41 42 44 45 46 47 48 49 51 52
 54 55 56 58 59 60 67 71 72 74 79 80 81 82 83 85 87 90 91 92 93 94 95 97
 98 78 96 99 77 39 50 43 65]
--------------------------------------------------
Weekly_Sales
359464
[24924.5  50605.27 13740.12 ... 56017.47  6817.48  1076.8 ]
--------------------------------------------------
Temperature
3528
[42.31 38.51 39.93 ... 75.87 77.55 74.09]
--------------------------------------------------
Fuel_Price
892
[2.572 2.548 2.514 2.561 2.625 2.667 2.72  2.732 2.719 2.77  2.808 2.795
 2.78  2.835 2.854 2.826 2.759 2.705 2.668 2.637 2.653 2.669 2.642 2.623
 2.608 2.64  2.627 2.692 2.664 2.619 2.577 2.565 2.582 2.624 2.603 2.633
 2.725 2.716 2.689 2

In [ ]:
for col in numerical_cols:
    px.histogram(df,x=col,title=f"Distribution of {col}").show()
    px.box(df,y=col,title=f"Boxplot of {col}").show()

### 2.3 Cleaning:

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         421570 non-null  int64         
 1   Dept          421570 non-null  int64         
 2   Date          421570 non-null  datetime64[ns]
 3   Weekly_Sales  421570 non-null  float64       
 4   IsHoliday     421570 non-null  bool          
 5   Temperature   421570 non-null  float64       
 6   Fuel_Price    421570 non-null  float64       
 7   MarkDown1     150681 non-null  float64       
 8   MarkDown2     111248 non-null  float64       
 9   MarkDown3     137091 non-null  float64       
 10  MarkDown4     134967 non-null  float64       
 11  MarkDown5     151432 non-null  float64       
 12  CPI           421570 non-null  float64       
 13  Unemployment  421570 non-null  float64       
 14  Type          421570 non-null  object        
 15  Size          421

In [52]:
missing_pct=(df.isna().mean()*100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
print("\n% missing per column:\n", missing_pct)


% missing per column:
 MarkDown2    73.611025
MarkDown4    67.984676
MarkDown3    67.480845
MarkDown1    64.257181
MarkDown5    64.079038
dtype: float64


In [53]:
fig_missing = px.bar(
    missing_pct, orientation="h",
    title="Missing values by column (%)",
    labels={"value": "% missing", "index": "column"},
)
fig_missing.show()
fig_missing.write_html("results/eda/missingness.html")
 

In [54]:
fig_target = px.histogram(
    df, x="Weekly_Sales", nbins=100,
    title="Weekly_Sales distribution",
)
fig_target.write_html("results/eda/target_distribution.html")

In [55]:
fig_target.show()

In [56]:
holiday_avg = df.groupby("IsHoliday")["Weekly_Sales"].mean().reset_index()
fig_holiday = px.bar(
    holiday_avg, x="IsHoliday", y="Weekly_Sales",
    title="Avg Weekly_Sales: Holiday vs Non-Holiday",
)
fig_holiday.write_html("results/eda/sales_by_holiday.html")
 

In [57]:
fig_holiday.show()

In [58]:
type_avg = df.groupby("Type")["Weekly_Sales"].mean().sort_values().reset_index()
fig_type = px.bar(
    type_avg, x="Type", y="Weekly_Sales",
    title="Avg Weekly_Sales by Store Type",
)
fig_type.write_html("results/eda/sales_by_store_type.html")

In [59]:
fig_type.show()

In [61]:
# we assume that null values in markdown columns mean no markdowns were applied, so we fill them with 0
markdown_cols=[c for c in df.columns if c.startswith('MarkDown')]
df[markdown_cols] = df[markdown_cols].fillna(0)

In [62]:
# CPI/Unemployment: forward-fill per store, using only PAST values in time
# order -> causally safe pre-split, no leakage from future/test rows.
df = df.sort_values(["Store", "Date"])
df[["CPI", "Unemployment"]] = df.groupby("Store")[["CPI", "Unemployment"]].ffill()

In [63]:
# Negative sales & outliers: kept deliberately (see EDA above) — real
# returns / real holiday spikes central to this project's thesis.
df["IsHoliday"] = df["IsHoliday"].astype(int)

In [64]:
with open("results/eda/cleaning_decisions.md", "w") as f:
    f.write(f"""# Cleaning Decisions
- MarkDown1-5 missing -> filled with 0 (no promo that week, domain fact).
- CPI/Unemployment missing -> forward-filled per store (slow-changing indicators).
- Negative Weekly_Sales ({n_negative} rows) -> kept, represent real returns.
- IQR outliers ({n_outliers} rows) -> kept, represent real holiday demand spikes.
""")
 
df.to_csv("data/processed/walmart_cleaned.csv", index=False)
print("\nSaved cleaned data + Plotly HTML figures to results/eda/")


Saved cleaned data + Plotly HTML figures to results/eda/


In [ ]:
from hijri_converter import Hijri

# Streamlit-style charts added before feature engineering
# Reproduce the market and seasonal sales insight in notebook form

def add_hijri_window_flags_for_eda(df):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])

    events = []
    for hijri_year in range(1430, 1435):
        events.append(("ramadan_start", pd.Timestamp(*Hijri(hijri_year, 9, 1).to_gregorian().datetuple())))
        events.append(("eid_fitr", pd.Timestamp(*Hijri(hijri_year, 10, 1).to_gregorian().datetuple())))
        events.append(("eid_adha", pd.Timestamp(*Hijri(hijri_year, 12, 10).to_gregorian().datetuple())))
    event_df = pd.DataFrame(events, columns=["event", "date"])

    for name, prefix in [("ramadan_start", "ramadan"), ("eid_fitr", "eid_fitr"), ("eid_adha", "eid_adha")]:
        ev_dates = event_df.loc[event_df["event"] == name, "date"]
        after = ev_dates[ev_dates >= df["Date"].min()]
        before = ev_dates[ev_dates <= df["Date"].max()]

        df[f"days_to_{prefix}"] = df["Date"].map(
            lambda d: (after[after >= d].min() - d).days if not after[after >= d].empty else 999
        )
        df[f"days_since_{prefix}"] = df["Date"].map(
            lambda d: (d - before[before <= d].max()).days if not before[before <= d].empty else 999
        )

    df["is_ramadan_window"] = ((0 <= df["days_since_ramadan"]) & (df["days_since_ramadan"] <= 29)) | ((0 <= df["days_to_ramadan"]) & (df["days_to_ramadan"] <= 6))
    df["is_pre_eid_window"] = ((0 <= df["days_to_eid_fitr"]) & (df["days_to_eid_fitr"] <= 6)) | ((0 <= df["days_to_eid_adha"]) & (df["days_to_eid_adha"] <= 6))
    return df

eda_df = add_hijri_window_flags_for_eda(df)
eda_df["season"] = np.where(eda_df["is_ramadan_window"] | eda_df["is_pre_eid_window"], "Ramadan/Eid window", "Regular period")

market_sales = eda_df.groupby("Store")["Weekly_Sales"].sum().reset_index().sort_values("Weekly_Sales", ascending=False)
fig_market = px.bar(market_sales, x="Store", y="Weekly_Sales", title="Total sales by market (Store)")
fig_market.show()
fig_market.write_html("results/eda/market_sales.html")

season_avg = eda_df.groupby(["Store", "season"])["Weekly_Sales"].mean().reset_index()
fig_season = px.bar(season_avg, x="Store", y="Weekly_Sales", color="season", barmode="group",
                   title="Avg Weekly_Sales: Ramadan/Eid window vs regular period, by market")
fig_season.show()
fig_season.write_html("results/eda/seasonal_comparison.html")

single_market = eda_df["Store"].sort_values().unique()[0]
trend = eda_df[eda_df["Store"] == single_market].sort_values("Date")
fig_trend = px.line(trend, x="Date", y="Weekly_Sales", title=f"Weekly sales over time — Store {single_market}")
for d in trend.loc[trend["is_ramadan_window"], "Date"]:
    fig_trend.add_vline(x=d, line_color="green", opacity=0.15)
fig_trend.show()
fig_trend.write_html("results/eda/sales_by_store_time.html")

overall_avg = eda_df.groupby("season")["Weekly_Sales"].mean()
diff_pct = (overall_avg["Ramadan/Eid window"] - overall_avg["Regular period"]) / overall_avg["Regular period"] * 100
print(f"Overall Ramadan/Eid impact vs regular period: {diff_pct:+.1f}%")

## Feature Engineering

In [65]:
import pandas as pd
from hijri_converter import Hijri
from sklearn.compose import ColumnTransformer # for preprocessing pipelines
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

In [66]:
df=pd.read_csv("data/processed/walmart_cleaned.csv",parse_dates=['Date'])
df

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,0,42.31,2.572,0.00,0.00,0.0,0.00,0.00,211.096358,8.106,A,151315
1,1,2,2010-02-05,50605.27,0,42.31,2.572,0.00,0.00,0.0,0.00,0.00,211.096358,8.106,A,151315
2,1,3,2010-02-05,13740.12,0,42.31,2.572,0.00,0.00,0.0,0.00,0.00,211.096358,8.106,A,151315
3,1,4,2010-02-05,39954.04,0,42.31,2.572,0.00,0.00,0.0,0.00,0.00,211.096358,8.106,A,151315
4,1,5,2010-02-05,32229.38,0,42.31,2.572,0.00,0.00,0.0,0.00,0.00,211.096358,8.106,A,151315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421565,45,93,2012-10-26,2487.80,0,58.85,3.882,4018.91,58.08,100.0,211.94,858.33,192.308899,8.667,B,118221
421566,45,94,2012-10-26,5203.31,0,58.85,3.882,4018.91,58.08,100.0,211.94,858.33,192.308899,8.667,B,118221
421567,45,95,2012-10-26,56017.47,0,58.85,3.882,4018.91,58.08,100.0,211.94,858.33,192.308899,8.667,B,118221
421568,45,97,2012-10-26,6817.48,0,58.85,3.882,4018.91,58.08,100.0,211.94,858.33,192.308899,8.667,B,118221


In [67]:
# hijri_converter implements the Umm al-Qura tabular calendar (the official
# Saudi calendar), so these dates are computed programmatically rather than
# typed in by hand. Covers a buffer around the dataset's Hijri years
# (~1431-1433H, corresponding to Gregorian 2010-2012).
event_specs = [("ramadan_start", 9, 1), ("eid_fitr", 10, 1), ("eid_adha", 12, 10)]
events = []
for hijri_year in range(1430, 1435):
    for event_name, month, day in event_specs:
        g = Hijri(hijri_year, month, day).to_gregorian()
        events.append((event_name, pd.Timestamp(g.year, g.month, g.day)))
hijri_events = pd.DataFrame(events, columns=["event", "date"])
print(hijri_events)

            event       date
0   ramadan_start 2009-08-22
1        eid_fitr 2009-09-20
2        eid_adha 2009-11-27
3   ramadan_start 2010-08-11
4        eid_fitr 2010-09-10
5        eid_adha 2010-11-16
6   ramadan_start 2011-08-01
7        eid_fitr 2011-08-30
8        eid_adha 2011-11-06
9   ramadan_start 2012-07-20
10       eid_fitr 2012-08-19
11       eid_adha 2012-10-26
12  ramadan_start 2013-07-09
13       eid_fitr 2013-08-08
14       eid_adha 2013-10-15


In [68]:
def add_days_to_from_event(df, events_df, event_name, col_prefix):
    """days_to_<event> (next occurrence) and days_since_<event> (last occurrence)."""
    ev = events_df.loc[events_df["event"] == event_name, ["date"]].sort_values("date")
    base = df[["Date"]].reset_index().rename(columns={"index": "_orig_idx"}).sort_values("Date")
 
    fwd = pd.merge_asof(base, ev.rename(columns={"date": "next_date"}),
                         left_on="Date", right_on="next_date", direction="forward")
    bwd = pd.merge_asof(base, ev.rename(columns={"date": "prev_date"}),
                         left_on="Date", right_on="prev_date", direction="backward")
 
    # restore original row order before extracting .values — merge_asof
    # needs Date-sorted input, so fwd/bwd come back in Date order, not the
    # original df order; skipping this silently misaligns every row.
    fwd = fwd.set_index("_orig_idx").sort_index()
    bwd = bwd.set_index("_orig_idx").sort_index()
 
    out = df.copy()
    out[f"days_to_{col_prefix}"] = (fwd["next_date"] - fwd["Date"]).dt.days.values
    out[f"days_since_{col_prefix}"] = (bwd["Date"] - bwd["prev_date"]).dt.days.values
    return out

In [69]:
 
df = add_days_to_from_event(df, hijri_events, "ramadan_start", "ramadan")
df = add_days_to_from_event(df, hijri_events, "eid_fitr", "eid_fitr")
df = add_days_to_from_event(df, hijri_events, "eid_adha", "eid_adha")
 
df["is_ramadan_window"] = df["days_since_ramadan"].between(0, 29) | df["days_to_ramadan"].between(0, 6)
df["is_pre_eid_window"] = df["days_to_eid_fitr"].between(0, 6) | df["days_to_eid_adha"].between(0, 6)
 
df.to_csv("data/processed/walmart_with_hijri_features.csv", index=False)
print("Feature engineering done. New columns:",
      [c for c in df.columns if "ramadan" in c or "eid" in c])

Feature engineering done. New columns: ['days_to_ramadan', 'days_since_ramadan', 'days_to_eid_fitr', 'days_since_eid_fitr', 'days_to_eid_adha', 'days_since_eid_adha', 'is_ramadan_window', 'is_pre_eid_window']


- PreProcessing

In [70]:
base_categorical = ["Store", "Dept", "Type"]
base_numerical = ["Temperature", "Fuel_Price", "CPI", "Unemployment", "IsHoliday"]
hijri_features = [
    "days_to_ramadan", "days_since_ramadan",
    "days_to_eid_fitr", "days_to_eid_adha",
    "is_ramadan_window", "is_pre_eid_window",
]
target = "Weekly_Sales"
df = df.dropna(subset=[target])  # only an unlabeled row must be dropped
 
 
def build_preprocessor(categorical_cols, numerical_cols):
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", SimpleImputer(strategy="median"), numerical_cols),  # safety net only
    ])
 
 
print("Preprocessing step ready. Feature sets defined:")
print(f"  baseline (no Hijri): {len(base_categorical + base_numerical)} columns")
print(f"  with Hijri features: {len(base_categorical + base_numerical + hijri_features)} columns")

Preprocessing step ready. Feature sets defined:
  baseline (no Hijri): 8 columns
  with Hijri features: 14 columns


## Step 5 : Modeling

In [71]:
#! pip install lightgbm

In [72]:
# %% [5] Modeling — self-contained: run this in a fresh kernel, after
# step_3_4_features_preprocessing.py has produced
# data/processed/walmart_with_hijri_features.csv
import pandas as pd
import plotly.express as px
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

def evaluate(pipeline, X_train, y_train, X_test, y_test, window_mask):
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    return {
        "overall_rmse": root_mean_squared_error(y_test, preds),
        "overall_mae": mean_absolute_error(y_test, preds),
        "overall_r2": r2_score(y_test, preds),
        "ramadan/eid_window_rmse": root_mean_squared_error(y_test[window_mask], preds[window_mask]),
    }, preds

In [73]:
# ---- Time-based split, reused everywhere below ----
split_date = df["Date"].quantile(0.8, interpolation="nearest")
train_df = df[df["Date"] <= split_date]
test_df = df[df["Date"] > split_date]
window_mask_test = (test_df["is_ramadan_window"] | test_df["is_pre_eid_window"]).values
print(f"Train: {train_df.shape}, Test: {test_df.shape}, split at {split_date.date()}")

X_train, y_train = train_df[hijri_cols], train_df[target]
X_test, y_test = test_df[hijri_cols], test_df[target].values

Train: (338738, 24), Test: (82832, 24), split at 2012-04-13


In [74]:
# ---- 5.1 Compare 3 model types, same features (with Hijri), same split.
# Each gets its OWN fresh estimator instance — a plain dict of unfitted
# estimators, never reused/refit across pipelines. ----
model_specs = {
    "LightGBM": LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42, verbosity=0),
    "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1),
}

results = {}
fitted_pipelines = {}
for name, estimator in model_specs.items():
    pipe = Pipeline([("preprocess", build_preprocessor(base_categorical, base_numerical + hijri_features)),
                      ("model", clone(estimator))])
    print(f"Training {name}...")
    metrics, _ = evaluate(pipe, X_train, y_train, X_test, y_test, window_mask_test)
    results[name] = metrics
    fitted_pipelines[name] = pipe  # each pipeline keeps its OWN fitted model, never shared

results_df = pd.DataFrame(results).T
best_model_name = results_df["overall_rmse"].idxmin()
print("\n", results_df)
print(f"\nBest model by overall RMSE: {best_model_name}")

fig_models = px.bar(results_df.reset_index().rename(columns={"index": "model"}),
                     x="model", y="overall_rmse",
                     title="Model comparison — overall RMSE (lower is better)", text_auto=".0f")
fig_models.write_html("results/eda/model_comparison_rmse.html")

fig_r2 = px.bar(results_df.reset_index().rename(columns={"index": "model"}),
                 x="model", y="overall_r2",
                 title="Model comparison — R² (higher is better)", text_auto=".3f")
fig_r2.write_html("results/eda/model_comparison_r2.html")
results_df.to_csv("results/model_comparison.csv")

Training LightGBM...
Training XGBoost...
Training RandomForest...

               overall_rmse  overall_mae  overall_r2  ramadan/eid_window_rmse
LightGBM       6669.067827  4392.842854    0.907832              6442.659595
XGBoost        7779.284698  5310.780633    0.874591              7830.205329
RandomForest   9921.643063  7300.482925    0.796007              9829.039069

Best model by overall RMSE: LightGBM


In [75]:
# ---- 5.2 Isolate the Hijri features' contribution, using the BEST model
# type only — a fresh clone for the baseline (no-Hijri) pipeline, so it
# never touches the already-fitted pipe_with_hijri. ----
best_spec = model_specs[best_model_name]

pipe_with_hijri = fitted_pipelines[best_model_name]  # already fitted above, reuse as-is
preds_with = pipe_with_hijri.predict(X_test)

pipe_no_hijri = Pipeline([("preprocess", build_preprocessor(base_categorical, base_numerical)),
                           ("model", clone(best_spec))])
results_baseline, preds_without = evaluate(pipe_no_hijri, train_df[baseline_cols], y_train,
                                            test_df[baseline_cols], y_test, window_mask_test)

comparison = pd.DataFrame([results_baseline, results[best_model_name]],
                           index=[f"{best_model_name}_no_hijri", f"{best_model_name}_with_hijri"])
comparison["window_rmse_improvement_%"] = (
    (comparison.loc[f"{best_model_name}_no_hijri", "ramadan/eid_window_rmse"] - comparison["ramadan/eid_window_rmse"])
    / comparison.loc[f"{best_model_name}_no_hijri", "ramadan/eid_window_rmse"] * 100
)
print("\n", comparison)
comparison.to_csv("results/baseline_vs_hijri_comparison.csv")

fig_hijri = px.bar(comparison.reset_index().rename(columns={"index": "version"}),
                    x="version", y="ramadan/eid_window_rmse",
                    title=f"Hijri features impact on {best_model_name} — Ramadan/Eid window RMSE",
                    text_auto=".0f")
fig_hijri.write_html("results/eda/hijri_impact.html")


                      overall_rmse  overall_mae  overall_r2  \
LightGBM_no_hijri     6869.385014  4505.489716    0.902212   
LightGBM_with_hijri   6669.067827  4392.842854    0.907832   

                     ramadan/eid_window_rmse  window_rmse_improvement_%  
LightGBM_no_hijri                6641.100240                   0.000000  
LightGBM_with_hijri              6442.659595                   2.988069  


In [76]:
# %% [5b] Per-department breakdown + feature importance
# Dept here is the RAW column from test_df, untouched by the encoder —
# the encoder only exists inside the pipeline's internal transform step.
dept_results = test_df[["Dept"]].copy()
dept_results["actual"] = y_test
dept_results["pred_with_hijri"] = preds_with
dept_results["pred_no_hijri"] = preds_without


def rmse_by_group(g, pred_col):
    return root_mean_squared_error(g["actual"], g[pred_col])


per_dept = dept_results.groupby("Dept").apply(
    lambda g: pd.Series({
        "rmse_no_hijri": rmse_by_group(g, "pred_no_hijri"),
        "rmse_with_hijri": rmse_by_group(g, "pred_with_hijri"),
        "n_rows": len(g),
    })
).reset_index()
per_dept["improvement_%"] = (
    (per_dept["rmse_no_hijri"] - per_dept["rmse_with_hijri"]) / per_dept["rmse_no_hijri"] * 100
)
per_dept = per_dept.sort_values("improvement_%", ascending=False)
print(per_dept.head(15))

fig_dept = px.bar(per_dept.head(20), x="Dept", y="improvement_%",
                   title="Hijri features: RMSE improvement % by Department (top 20)")
fig_dept.write_html("results/eda/improvement_by_dept.html")
per_dept.to_csv("results/improvement_by_dept.csv", index=False)

    Dept  rmse_no_hijri  rmse_with_hijri  n_rows  improvement_%
14    16   14413.894478     10032.845562  1260.0      30.394623
16    18    8887.193159      7185.723236   851.0      19.145189
60    72   13439.876402     11099.379056  1188.0      17.414575
0      1    5417.511654      4608.282827  1260.0      14.937279
62    77    6912.508792      5913.706947    12.0      14.449195
65    80    6484.403457      5604.850834  1177.0      13.564126
52    55    4344.995468      3793.098844  1074.0      12.701892
4      5    6386.684510      5585.844574  1260.0      12.539212
53    56    7587.901400      6819.222865  1180.0      10.130318
37    39   10674.549012      9595.188279     9.0      10.111535
43    45    4219.901272      3824.879401   182.0       9.360927
24    26    3678.918868      3352.100466  1138.0       8.883545
30    32    3247.565666      2990.917365  1217.0       7.902790
41    43    4553.214817      4196.391202     1.0       7.836740
69    85    3526.260450      3259.153548

C:\Users\SOMIA\AppData\Local\Temp\ipykernel_5408\3713262582.py:14: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [77]:
# Feature importance — pipe_with_hijri was fitted once, above, and never
# reused/refit since, so its preprocessor's feature names and its model's
# feature_importances_ are guaranteed to correspond to the same fit.
feature_names = pipe_with_hijri.named_steps["preprocess"].get_feature_names_out()
importances = pipe_with_hijri.named_steps["model"].feature_importances_
assert len(feature_names) == len(importances), (
    f"Mismatch: {len(feature_names)} feature names vs {len(importances)} importances — "
    "the pipeline's model was refit or replaced after this point."
)

importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df = importance_df.sort_values("importance", ascending=False).head(20)
importance_df["is_hijri_feature"] = importance_df["feature"].str.contains("ramadan|eid", case=False)

fig_importance = px.bar(importance_df, x="importance", y="feature", orientation="h",
                         color="is_hijri_feature",
                         title=f"{best_model_name} — top 20 feature importances (Hijri features highlighted)")
fig_importance.update_layout(yaxis={"categoryorder": "total ascending"})
fig_importance.write_html("results/eda/feature_importance.html")
print("\n", importance_df)



                    feature  importance  is_hijri_feature
118           cat__Dept_92         309             False
9            cat__Store_10         282             False
105           cat__Dept_72         282             False
137  num__days_to_eid_adha         281              True
121           cat__Dept_95         281             False
81            cat__Dept_38         236             False
46             cat__Dept_2         228             False
128            cat__Type_C         219             False
134   num__days_to_ramadan         218              True
127            cat__Type_B         209             False
131               num__CPI         204             False
126            cat__Type_A         202             False
83            cat__Dept_40         188             False
116           cat__Dept_90         171             False
32           cat__Store_33         167             False
51             cat__Dept_7         152             False
13           cat__Store_14   

In [78]:
import joblib
os.makedirs("models", exist_ok=True)
joblib.dump(pipe_with_hijri, "models/crescentiq_pipeline.pkl")
joblib.dump(hijri_cols, "models/crescentiq_feature_columns.pkl")
print("\nSaved trained pipeline to models/crescentiq_pipeline.pkl")


Saved trained pipeline to models/crescentiq_pipeline.pkl
